# 05 · Initialization: keeping signals alive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/05_initialization.ipynb) &nbsp; 📖 Concept deck: [Initialization](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/06-initialization/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** Xavier (Glorot) and He (Kaiming) initialisation, and measurements of how activation scale explodes or vanishes through depth when the variance is wrong.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

## Task 1: Xavier and He standard deviations

Return the weight standard deviation for a layer with `fan_in` inputs and `fan_out` outputs:
`'xavier'`: $\sqrt{2/(\text{fan\_in} + \text{fan\_out})}$ (Glorot & Bengio, 2010); `'he'`: $\sqrt{2/\text{fan\_in}}$ (He et al., 2015); `'naive'`: 1.

In [ ]:
def init_std(kind, fan_in, fan_out):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_init_std(f):
    assert np.isclose(f('xavier', 100, 300), np.sqrt(2 / 400))
    assert np.isclose(f('he', 512, 10), np.sqrt(2 / 512))
    assert f('naive', 5, 5) == 1

check("init_std", _test_init_std, init_std)

## Task 2: Activation scale through 50 layers

Push 512 standard-normal inputs through `depth` layers of width 256: $h \leftarrow \text{act}(h W)$ with W drawn using `init_std(kind, 256, 256)`. `act` is `'relu'` or `'tanh'`.
Return the array of activation standard deviations after each layer.

In [ ]:
def activation_scale(kind, act, depth=50, width=256, seed=0):
    r = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_activation_scale(f):
    s = f('he', 'relu')
    assert len(s) == 50 and 0.3 < s[-1] / s[0] < 3, "He init keeps ReLU activations at a stable scale"
    x = f('xavier', 'relu')
    assert x[-1] < 1e-3 * x[0], "Xavier is too small for ReLU (half the units are zero): the signal halves in variance each layer"
    n = f('naive', 'relu', depth=10)
    assert n[-1] > 1e8, "std 1 weights blow up by ≈ √(width/2) per layer"

check("activation_scale", _test_activation_scale, activation_scale)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Xavier and He standard deviations
def init_std_solution(kind, fan_in, fan_out):
    return {'xavier': np.sqrt(2 / (fan_in + fan_out)), 'he': np.sqrt(2 / fan_in), 'naive': 1.0}[kind]

check("init_std (reference)", _test_init_std, init_std_solution)

In [ ]:
# @title Solution for Task 2: Activation scale through 50 layers
def activation_scale_solution(kind, act, depth=50, width=256, seed=0):
    r = np.random.default_rng(seed); h = r.normal(size=(512, width)); out = []
    fn = (lambda z: np.maximum(z, 0)) if act == 'relu' else np.tanh
    for _ in range(depth):
        h = fn(h @ r.normal(scale=init_std_solution(kind, width, width), size=(width, width))); out.append(h.std())
    return np.array(out)

check("activation_scale (reference)", _test_activation_scale, activation_scale_solution)

---
## Experiments
**Summary table.** Final-layer activation std after 50 layers for each initialisation and activation. Watch tanh with std-1 weights: its std stays near 1 because every unit is saturated at ±1, where the derivative is ≈ 0, so the signal survives but gradients do not.

In [ ]:
for act in ['relu', 'tanh']:
    for kind in ['naive', 'xavier', 'he']:
        with np.errstate(over='ignore', invalid='ignore'):
            s = activation_scale_solution(kind, act)
        print(f"{act:5s} {kind:7s}: std after layer 1 = {s[0]:.3g}, after layer 50 = {s[-1]:.3g}")

---
## Takeaways
* Each layer multiplies activation variance by about fan_in × Var(w) × (activation's variance factor).
* Choose Var(w) to make that factor 1: Xavier for tanh-like activations, He (2/fan_in) for ReLU.
* Normalisation layers and residual connections (next deck) make deep transformers far less sensitive to this.